# Setup

In [1]:
# set parameters
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
import plotly.graph_objects as go
import sys
sys.path.append('../../assets/python/')
import dmg5e
import estats5e
import tfb

METADATA = {'Contributor': 'T. Dunn'}
SAVEFIGS = False

In [3]:
# functions and classes
import json
import numpy as np
import re
import uuid

COLOR_LIST = [
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
    # repeat
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
]

class MyEncoder(json.JSONEncoder):
    def default(self, o):
        return o.__dict__

class EncounterLibrary:
    def __init__(self, encounters=[], file=None):
        """
        Constructs a new encounter library.

        Parameters
        ----------
        encounters : list
            A list of encounters.
        
        file : str
            If provided then the encounters will be loaded from file.
        """
        if encounters:
            self.set_encounters(encounters)
        else:
            if file:
                self.from_json_file(file)

    def __repr__(self):
        return f'{self.__dict__}'

    def from_json_file(self, file):
        """
        Loads encounters from a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        """
        with open(file, 'r') as fin:
            self.set_encounters(json.load(fin))
    
    def to_json_file(self, file, compact_lists=False, **kwargs):
        """
        Saves encounters as a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        
        compact_lists : bool
            If ``True`` then lists will be written in a compact form with one entry per line.
            Default value ``False``.
        """
        s = json.dumps(self.encounters, cls=MyEncoder, **kwargs)
        if compact_lists:
            s = re.sub(r'(\[)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1\2', s)
            s = re.sub(r'(,)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1 \2', s)
            s = re.sub(r'(,[\s\n]+[^:,\[\]]+?)[\s\n]+(?=\])', r'\1', s)
        
        with open(file, 'w') as fout:
            fout.write(s)
    
    def get_encounters(self, book_paths=[], ids=None):
        """
        Returns a filtered list of encounters.

        Parameters
        ----------
        book_path : str
            A regular expression to be applied to encounter ``book_path`` attribute using the ``re.match`` function.
        
        Returns
        -------
        encounters : list
            The encounters that matched the given book_path regular expression.
        """    
        encounter_ids = ids if ids else []
        
        for book_path in book_paths:
            encounter_ids.extend([e.id for e in self.encounters if re.match(book_path, e.book_path)])
        
        return [e for e in self.encounters if e.id in encounter_ids]

    def set_encounters(self, encounters):
        self.encounters = []
        for e in encounters:
            if type(e) is dict:
                self.encounters.append(Encounter(**e))
            else:
                self.encounters.append(e)

class Encounter:
    def __init__(self, **kwargs):
        self.id = kwargs.get('id', str(uuid.uuid4()))
        self.type = kwargs.get('type', None)
        self.book_path = kwargs.get('book_path', None)
        self.allies = kwargs.get('allies', [])
        self.bystanders = kwargs.get('bystanders', [])
        self.enemies = kwargs.get('monsters', [])
        self.enemies = kwargs.get('enemies', self.enemies)

    def __repr__(self):
            return f'{self.__dict__}'

    def book(self):
        return self.book_path.split('; ')[0]

    def allies_xp_values(self):
        """
        Returns a list of XP values for the ally monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each ally monster in the encounter.
        """
        xp_vals = []
        for monster in self.allies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals

    def enemy_cr_values_from_xp(self):
        """
        Returns a list of CR values for the enemy monsters in the encounter.

        These CR values are determined from the monsters' XP values, which may differ from the default values for each CR.

        Returns
        ----------
        cr_vals : list
            The CR values for each enemy monster in the encounter.
        """
        def _find_nearest_loc(array, value):
            array = np.asarray(array)
            idx = (np.abs(array - value)).argmin()
            return idx
        
        cr_vals = []
        for xp in self.enemy_xp_values():
            id = _find_nearest_loc(dmg5e.MONSTER_DEFAULTS['XP'], xp)
            cr_vals.append(dmg5e.MONSTER_DEFAULTS['CR'][id])
        return cr_vals

    def enemy_monster_ids(self):
        """
        Returns a list of monster Ids for the enemy monsters in the encounter.

        Returns
        ----------
        ids : list
            The monster ids for each enemy monster in the encounter.
        """
        ids = []
        for monster in self.enemies:
            ids.extend(monster[0]*[monster[1]])
        
        return ids
    
    def enemy_xp_values(self):
        """
        Returns a list of XP values for the enemy monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each enemy monster in the encounter.
        """
        xp_vals = []
        for monster in self.enemies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals

    def enemy_xp_total(self):
        """
        Returns the total XP value of all enemy monsters in the encounter.

        Returns
        ----------
        xp_total : float
            The total of all enemy monster XP values in the encounter.
        """
        
        return sum(self.enemy_xp_values())

    def adjusted_enemy_xp_total(self, pc_levels, rules='2014'):
        """
        Returns the adjusted XP total for the encounter.

        Parameters
        ----------
        pc_levels : list
            The level of each PC in the encounter.
        
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        ----------
        xp_total : float
            The adjusted total XP for the encounter.
        """
        if rules == '2014':
            em = dmg5e.encounter_xp_multiplier(len(pc_levels), len(self.enemy_xp_values()))
            return em*self.enemy_xp_total()
        elif rules == '2024':
            return self.enemy_xp_total()
        elif rules == 'tfb':
            xp_vals = self.enemy_xp_values()
            xp_total = sum(xp_vals)
            xp_square = sum(np.sqrt(xp_vals))**2
            xp_multi = 0.5*(xp_square - xp_total)
            return xp_total + xp_multi*0.3

class Party:
    def __init__(self, levels=[]):
        self.levels = levels
    
    def xp_budget(self, rules='2014'):
        """
        Returns the adventuring day XP budget for the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        xp_budget : float
            The party's adventuring day XP budget.
        """
        return sum(self.pc_xp_budgets(rules))
    
    def xp_thresholds(self, rules='2014'):
        """
        Returns encounter XP thresholds for the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        party_xps : dict
            A dict of encounter difficulties and the party's corresponding XP thresholds.
        """
        pc_xps = self.pc_xp_thresholds(rules)
        party_xps = {}
        for diff in pc_xps:
            party_xps[diff] = sum(pc_xps[diff])
        
        return party_xps
    
    def pc_xp_budgets(self, rules='2014'):
        """
        Returns the adventuring day XP budget for each PC in the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        xp_budgets : list
            A list containing the adventuring day XP budgets for each PC in the party.
        """
        if rules == '2014':
            xp_budget = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
        else:
            xp_budget = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
        
        return [xp_budget[lvl-1] for lvl in self.levels]

    def pc_xp_thresholds(self, rules='2014'):
        """
        Returns encounter XP thresholds for each PC in the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        pc_xps : dict
            A dict of encounter difficulties, containing a lists of XP thresholds for each PC in the party.
        """
        if rules == '2014':
            xp_thresholds = {
                'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
                'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
                'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
                'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
                'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
                'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
            }
        else:
            xp_thresholds = {
                'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
                'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
                'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
            }
        
        pc_xps = {}
        for diff in xp_thresholds:
            pc_xps[diff] = [xp_thresholds[diff][lvl-1] for lvl in self.levels]
        return pc_xps

def player_character_xp_budget(pc_level, rules='2014'):
    """Returns the adventuring day XP budget for a single PC of the given level.
    """
    if rules == '2014':
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    else:
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    return XP_BUDGET[pc_level-1]

def player_character_xp_thresholds(pc_level, rules='2014'):
    """Returns the encounter XP thresholds for each encounter difficulty for a PC of the given level.
    """
    if rules == '2014':
        XP_THRESHOLDS = {
            'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
            'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
            'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
            'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
            'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
            'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
        }
    else:
        XP_THRESHOLDS = {
            'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
            'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
            'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
        }
    pc_xps = {}
    for diff in XP_THRESHOLDS:
        pc_xps[diff] = XP_THRESHOLDS[diff][pc_level-1]
    return pc_xps

def party_xp_budget(levels, rules='2014'):
    """Calculate the adventuring day XP budget for a party of PCs with the given levels.
    """
    # calculates the XP budget for a party of PCs
    return sum([player_character_xp_budget(lvl, rules=rules) for lvl in levels])

def party_xp_thresholds(levels, rules='2014'):
    """calculates the XP thresholds for a party of PCs based on their levels and the rules set being used
    """
    party_xps = {}
    for lvl in levels:
        pc_xps = player_character_xp_thresholds(lvl, rules=rules)
        for diff, xp in pc_xps.items():
            party_xps[diff] = party_xps.get(diff, 0) + xp
    
    return party_xps

def encounter_multiplier_DMG(pc_count, npc_count):
    """Returns the encounter multiplier given by the 2014 DMG
    pc_count -- number of PCs in the encounter
    npc_count -- number of NPCs in the encounter
    """
    n_array = np.asarray([1,2,3,7,11,15])
    m_array = np.asarray([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
    i = 1 + n_array[n_array <= max(npc_count,1)].argmax()
    if pc_count >= 6:
        i -= 1
    elif pc_count <= 2:
        i += 1
    return m_array[i]

def encounter_difficulty(pc_thresholds, encounter_xp, rules='2014'):
    """Determines the encounter's difficulty category by comparing its XP value against the party's XP thresholds.
    """
    difficulties = list(pc_thresholds.keys())
    xp_values = list(pc_thresholds.values())
    indx = np.argsort(xp_values)
    
    if rules == '2014':
        difficulty = 'Trivial'
        for i in indx:
            if encounter_xp >= xp_values[i]:
                difficulty = difficulties[i]
    elif rules == '2024':
        difficulty = 'Very High'
        for i in reversed(indx):
            if encounter_xp <= xp_values[i]:
                difficulty = difficulties[i]
    return difficulty


def _find_nearest_loc(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

def monster_challenge_rating_from_xp(xp):
    id = _find_nearest_loc(dmg5e.MONSTER_DEFAULTS['XP'], xp)
    return dmg5e.MONSTER_DEFAULTS['CR'][id]

# Load dataset

In [4]:
# Construct data frame
import numpy as np
import pandas as pd
import os

book_dict = {
    "Tyranny of Dragons": {'acronym': 'ToD', 'file': 'tod.json', 'type': 'campaign', 'date': '8/19/2014'},
    "Princes of the Apocalypse": {'acronym': 'PotA', 'file': 'pota.json', 'type': 'campaign', 'date': '4/7/2015'},
    "Curse of Strahd": {'acronym': 'CoS', 'file': 'cos.json', 'type': 'campaign', 'date': '3/16/2016'},
    "Storm King's Thunder": {'acronym': 'SKT', 'file': 'skt.json', 'type': 'campaign', 'date': '9/6/2016'},
    "Tales from the Yawning Portal": {'acronym': 'TftYP', 'file': 'tftyp.json', 'type': 'anthology', 'date': '4/4/2017'},
    "Tomb of Annihilation": {'acronym': 'ToA', 'file': 'toa.json', 'type': 'campaign', 'date': '9/19/2017'},
    "Waterdeep: Dragon Heist": {'acronym': 'W:DH', 'file': 'wdh.json', 'type': 'campaign', 'date': '9/18/2018'},
    "Waterdeep: Dungeon of the Mad Mage": {'acronym': 'W:DotMM', 'file': 'wdotmm.json', 'type': 'campaign', 'date': '11/20/2018'},
    "Ghosts of Saltmarsh": {'acronym': 'GoS', 'file': 'gos.json', 'type': 'anthology', 'date': '5/21/2019'},
    "Baldur’s Gate: Descent into Avernus": {'acronym': 'BG:DiA', 'file': 'bgdia.json', 'type': 'campaign', 'date': '9/17/2019'},
    "Icewind Dale: Rime of the Frostmaiden": {'acronym': 'ID:RotF', 'file': 'idrotf.json', 'type': 'campaign', 'date': '9/15/2020'},
    "Candlekeep Mysteries": {'acronym': 'CM', 'file': 'cm.json', 'type': 'anthology', 'date': '3/16/2021'},
    "Critical Role: Call of the Netherdeep": {'acronym': 'CR:CotN', 'file': 'cotn.json', 'type': 'campaign', 'date': '3/15/2022'},
    "Journeys through the Radiant Citadel": {'acronym': 'JttRC', 'file': 'jttrc.json', 'type': 'anthology', 'date': '7/19/2022'},
    "Dragonlance: Shadow of the Dragon Queen": {'acronym': 'D:SotDQ', 'file': 'sotdq.json', 'type': 'campaign', 'date': '12/6/2022'},
    "Keys from the Golden Vault": {'acronym': 'KftGV', 'file': 'kftgv.json', 'type': 'anthology', 'date': '2/21/2023'},
    "Phandelver and Below: The Shattered Obelisk": {'acronym': 'PaB:TSO', 'file': 'pbtso.json', 'type': 'campaign', 'date': '9/19/2023'},
    "Vecna: Eve of Ruin": {'acronym': 'V:EoR', 'file': 'veor.json', 'type': 'campaign', 'date': '5/21/2024'},
    "Quests from the Infinite Staircase": {'acronym': 'QftIS', 'file': 'qftis.json', 'type': 'anthology', 'date': '7/16/2024'},
    "Dragon Delves": {'acronym': 'DD', 'file': 'drde.json', 'type': 'anthology', 'date': '7/8/2025'},
}

# load adventures into encounter library
encounters = []
encounters_path = '../../assets/data/encounters'
for book in book_dict:
    file = book_dict[book]['file']
    elib = EncounterLibrary(file=os.path.join(encounters_path, file))
    encounters.extend(elib.encounters)

elib = EncounterLibrary(encounters=encounters)

# load campaign information
with open('../../assets/data/encounters/campaigns.json', 'r') as fin:
    campaigns = json.load(fin)

encounters = []
for campaign in campaigns:
    for group in campaign['groups']:
        encs = elib.get_encounters(ids=group['encounter_ids'])
        party = Party(group['party'])
        for enc in encs:
            if enc.type != 'combat': continue
            e = {}
            e['id'] = enc.id
            e['type'] = enc.type
            e['book'] = enc.book()
            e['book_acronym'] = book_dict.get(enc.book(), {}).get('acronym', '')
            e['adventure'] = enc.book_path.split('; ')[1]
            e['monsters'] = enc.enemy_monster_ids()
            e['monsters_xp'] = enc.enemy_xp_values()
            #e['monsters_cr'] = [monster_challenge_rating_from_xp(xp) for xp in enc.enemy_xp_values()]
            e['monsters_cr'] = enc.enemy_cr_values_from_xp()
            e['n_monsters'] = len(enc.enemy_xp_values())
            e['n_unique_monsters'] = len(np.unique(enc.enemy_monster_ids()))
            e['party'] = group['party']
            e['XP_total'] = enc.enemy_xp_total()
            e['PC_XP_total'] = e['XP_total']/len(group['party'])
            try:
                #e['2014 adj_XP_total'] = enc.adjusted_enemy_xp_total(group['party'], rules='2014')
                e['2014 adj_XP_total'] = enc.adjusted_enemy_xp_total(party.levels, rules='2014')
                e['2014 party_xp_thresholds'] = party_xp_thresholds(group['party'], rules='2014')
                e['2014 party_xp_budget'] = party_xp_budget(group['party'], rules='2014')
                e['2014 difficulty'] = encounter_difficulty(e['2014 party_xp_thresholds'], e['2014 adj_XP_total'], rules='2014')

                e['2024 adj_XP_total'] = enc.adjusted_enemy_xp_total(group['party'], rules='2024')
                e['2024 party_xp_thresholds'] = party_xp_thresholds(group['party'], rules='2024')
                e['2024 party_xp_budget'] = party_xp_budget(group['party'], rules='2024')
                e['2024 difficulty'] = encounter_difficulty(e['2024 party_xp_thresholds'], e['2024 adj_XP_total'], rules='2024')

                e['TFB adj_XP_total'] = enc.adjusted_enemy_xp_total(group['party'], rules='tfb')
                e['TFB party_xp_thresholds'] = party_xp_thresholds(group['party'], rules='2014')
                e['TFB party_xp_budget'] = party_xp_budget(group['party'], rules='2014')
                e['TFB difficulty'] = encounter_difficulty(e['TFB party_xp_thresholds'], e['TFB adj_XP_total'], rules='2014')
                encounters.append(e)
            except:
                #print(f"{enc['book_path']}:")
                #print(f"  monsters: {enc['monsters']}")
                pass

print(f'total encounters: {len(encounters)}')

df = pd.DataFrame({
    'id': [e['id'] for e in encounters],
    'book': [e['book'] for e in encounters],
    'book_acronym': [e['book_acronym'] for e in encounters],
    'adventure': [e['adventure'] for e in encounters],
    'type': [e['type'] for e in encounters],
    'party': [e['party'] for e in encounters],
    'party_level': [np.mean(e['party']) for e in encounters],
    'monsters': [e['monsters'] for e in encounters],
    'monsters_xp': [e['monsters_xp'] for e in encounters],
    'monsters_cr': [e['monsters_cr'] for e in encounters],
    'n_monsters': [e['n_monsters'] for e in encounters],
    'n_unique_monsters': [e['n_unique_monsters'] for e in encounters],
    'XP_total': [e['XP_total'] for e in encounters],
    'PC_XP_total': [e['PC_XP_total'] for e in encounters],

    '2014 adj_XP_total': [e['2014 adj_XP_total'] for e in encounters],
    '2014 party_XP_budget': [e['2014 party_xp_budget'] for e in encounters],
    '2014 difficulty': [e['2014 difficulty'] for e in encounters],

    '2024 adj_XP_total': [e['2024 adj_XP_total'] for e in encounters],
    '2024 party_XP_budget': [e['2024 party_xp_budget'] for e in encounters],
    '2024 difficulty': [e['2024 difficulty'] for e in encounters],

    'TFB adj_XP_total': [e['TFB adj_XP_total'] for e in encounters],
    'TFB party_XP_budget': [e['TFB party_xp_budget'] for e in encounters],
    'TFB difficulty': [e['TFB difficulty'] for e in encounters],
})
book_categories = list(book_dict.keys())
book_acronym_categories = [v['acronym'] for v in book_dict.values()]
df['book'] = df['book'].astype('category')
df['book'] = df['book'].cat.set_categories(book_categories, ordered=True)
df['book_acronym'] = df['book_acronym'].astype('category')
df['book_acronym'] = df['book_acronym'].cat.set_categories(book_acronym_categories, ordered=True)
df['2014 difficulty'] = df['2014 difficulty'].astype('category')
df['2014 difficulty'] = df['2014 difficulty'].cat.set_categories(['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly'], ordered=True)
df['2014 XP_ratio'] = 2*df['2014 adj_XP_total']/df['2014 party_XP_budget']
df['2014 XP_mult'] = df['2014 adj_XP_total']/df['XP_total']
df['2024 difficulty'] = df['2024 difficulty'].astype('category')
df['2024 difficulty'] = df['2024 difficulty'].cat.set_categories(['Low','Moderate','High','Very High'], ordered=True)
df['2024 XP_ratio'] = 2*df['2024 adj_XP_total']/df['2024 party_XP_budget']
df['2024 XP_mult'] = df['2024 adj_XP_total']/df['XP_total']
df['TFB difficulty'] = df['TFB difficulty'].astype('category')
df['TFB difficulty'] = df['TFB difficulty'].cat.set_categories(['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly'], ordered=True)
df['TFB XP_ratio'] = 2*df['TFB adj_XP_total']/df['TFB party_XP_budget']
df['TFB XP_mult'] = df['TFB adj_XP_total']/df['XP_total']

total encounters: 3731


# Figures

In [4]:
# Fig. 1: histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dfG = df[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

"""fig.add_trace(go.Bar(
    x=dfG[f'{rules} difficulty'],
    y=dfG[f'{rules} XP_ratio']/dfG[f'{rules} XP_ratio'].sum(), 
    hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    #xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.35], tickformat='.0%', dtick=0.05, minor_dtick=0.01),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.write_image('../../assets/images/adventure-encounter-difficulties.png')
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-2014-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-2014-small')

In [27]:
# Fig. 2: Plots a histogram of encounter XPs using 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()
ymax = 0.16
fig.add_trace(go.Histogram(
    x=df[f'{rules} XP_ratio'], 
    xbins_size=0.05, 
    histnorm='probability', 
    showlegend=False,
    hovertemplate='XP ratio %{x:.2f}<br>probability %{y:.1%}<extra></extra>',
))

if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[diff, diff+dx], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=diff + dx/2, 
        y=0.9*ymax,
        text=name,
        showarrow=False,
        font_size=12,
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='XP ratio', range=[0,1.5], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='encounters [%]', range=[0,ymax], tickformat='.0%', dtick=0.02, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-pdf-2014-large', style='width: 600px;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-pdf-2014-small', style='width: 600px;')

In [55]:
# Fig. 3: plots the excounter XP distribution after normalizing for XP budget
import plotly.graph_objects as go
import numpy as np

rules = '2014'

fig = go.Figure()

dft = df[df['2014 XP_ratio'].between(0, 2)]
vals, edges = np.histogram(dft['2014 XP_ratio'], 40)

x = np.convolve(edges, np.ones(2)/2, mode='valid')
y = vals/sum(vals)

fig.add_trace(go.Bar(
    x=x, 
    y=y*x/np.trapezoid(y*x), 
    showlegend=False,
    marker_line_width=0,
    hovertemplate='XP ratio %{x:.2f}<br>selection probability %{y:.1%}<extra></extra>',
))

ymax = 0.09
if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[diff, diff+dx], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=diff + dx/2, 
        y=0.9*ymax,
        text=name,
        showarrow=False,
        font_size=12,
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='group',
    bargap=0,
    xaxis=dict(title_text='XP ratio', range=[0,1.5], dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='selection probability', range=[0,ymax], tickformat='.2f'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-selection-pdf-2014-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-selection-pdf-2014-small')

In [ ]:
# Fig. 4: mean encounter difficulty by level
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

xmax = 24
if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[0, xmax], 
        y=[diff+dx, diff+dx], 
        mode='none',
        fill='tonexty', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=0.92*xmax, 
        y=diff + dx/2,
        text=name,
        showarrow=False,
        font_size=12,
    )

dft = df[df[f'{rules} difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['party_level',f'{rules} XP_ratio']].groupby(['party_level'], observed=True).median().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG[f'{rules} XP_ratio'], 
    mode='markers+lines',
    line_color=COLOR_LIST[0],
    name='all encounters',
    hovertemplate='<b>all encounters</b><br>level %{x}<br>XP ratio %{y:.2f}<extra></extra>',
))

dft = df[df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['party_level',f'{rules} XP_ratio']].groupby(['party_level'], observed=True).median().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG[f'{rules} XP_ratio'], 
    mode='markers+lines',
    line_color=COLOR_LIST[1],
    name='non-Trivial encounters',
    hovertemplate='<b>non-Trivial encounters</b><br>level %{x}<br>XP ratio %{y:.2f}<extra></extra>',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #margin=dict(b=110),
    #barmode='stack',
    xaxis=dict(title_text='level', range=[0,24], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='median XP ratio', range=[0,1], dtick=0.2, minor_dtick=0.05),
    legend=dict(xanchor='left', x=0.02, yanchor='top', y=1.00, bgcolor='rgba(0,0,0,0)'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-median-xp-ratio-by-level-2014-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-median-xp-ratio-by-level-2014-small')

In [ ]:
# Fig. 5: Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]


if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[-1, len(books)+2], 
        y=[diff+dx, diff+dx],
        mode='none',
        fill='tonexty', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=len(books)+1, 
        y=diff + dx/2,
        text=name,
        showarrow=False,
        font_size=12,
    )

encounters_mean = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].median()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounters_mean.append(xr_mean)

fig.add_trace(go.Scatter(
    x=list(range(len(encounters_mean))), 
    y=encounters_mean,
    mode='markers+lines',
    name='all encounters',
    line_color=COLOR_LIST[0],
    hovertemplate='<b>all encounters</b><br>%{x}<br>XP ratio %{y:.2f}<extra></extra>',
))

encounters_mean = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].median()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounters_mean.append(xr_mean)

fig.add_trace(go.Scatter(
    x=list(range(len(encounters_mean))), 
    y=encounters_mean,
    mode='markers+lines',
    name='non-Trivial encounters',
    line_color=COLOR_LIST[1],
    hovertemplate='<b>non-Trivial encounters</b><br>%{x}<br>XP ratio %{y:.2f}<extra></extra>',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='median XP ratio', range=[0,1.0], automargin=True, tickformat='.1f', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.02, yanchor='top', y=1.00, bgcolor='rgba(0,0,0,0)'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-median-xp-ratio-by-book-large', style='aspect-ratio: 600/500;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-median-xp-ratio-by-book-small', style='aspect-ratio: 600/500;')

In [ ]:
# Fig. 6: Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

encounter_std = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounter_std.append(xr_std)

fig.add_trace(go.Scatter(
    x=books, 
    y=encounter_std,
    mode='markers+lines',
    #showlegend=False,
    name='all encounters',
    hovertemplate='<b>all encounters</b><br>%{x}<br>XP ratio sigma %{y:.2f}<extra></extra>',
))

encounter_std = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounter_std.append(xr_std)

fig.add_trace(go.Scatter(
    x=books, 
    y=encounter_std,
    mode='markers+lines',
    #showlegend=False,
    name='non-Trivial encounters',
    hovertemplate='<b>non-Trivial encounters</b><br>%{x}<br>XP ratio sigma %{y:.2f}<extra></extra>',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio sigma', range=[0,2.0], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.02, yanchor='top', y=1.00, bgcolor='rgba(0,0,0,0)'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-std-xp-ratio-by-book-large', style='aspect-ratio: 600/500;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-std-xp-ratio-by-book-small', style='aspect-ratio: 600/500;')

In [ ]:
# Fig. 7: histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale
from plotly.subplots import make_subplots

rules = '2014'

diff_list = list(df[f'{rules} difficulty'].cat.categories)

# pre-2022
dft = df[~df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

data1 = []
n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    data1.append(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

# post-2022
dft = df[df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

data2 = []
n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    data2.append(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

# create large figure
fig_large = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=['pre-2022','post-2022'])
fig_large.add_traces(data1, rows=1, cols=1)
fig_large.add_traces(data2, rows=1, cols=2)

fig_large.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict( automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.4], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis2=dict(dtick=0.1, minor_dtick=0.05),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=800, 
    height=450,
)

for a in fig_large.layout.annotations:
    a.update(y=a['y']-0.08)

fig_large.show(config=tfb.FIG_CONFIG)

# create small figure
fig_small = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=['pre-2022','post-2022'])
fig_small.add_traces(data1, rows=1, cols=1)
fig_small.add_traces(data2, rows=2, cols=1)

fig_small.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict( automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.4], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis2=dict(title_text='encounters [%]', range=[0,0.4], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=550, 
    height=800,
)

for a in fig_small.layout.annotations:
    a.update(y=a['y']-0.04)

fig_small.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig_large.update_layout(autosize=True, width=None, height=None)
    fig_small.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig_large, format='large', name=f'./fig-encounter-difficulty-split-2014-large', style='width:800px; min-width:50%; max-width:100%; max-height:450px; min-height:300; aspect-ratio: 800/450;')
    tfb.save_fig_html(fig_small, format='small', name=f'./fig-encounter-difficulty-split-2014-small', style='width:550px; min-width:50%; max-width:100%; max-height:800px; min-height:600; aspect-ratio: 550/800;')

# Unused

## Number of Encounters

In [390]:
# histogram of total encounters by book
import plotly.graph_objects as go

dfG = df[['book_acronym','party']].groupby('book_acronym', observed=True).count()
books = [v['acronym'] for k, v in book_dict.items()]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=books,
    y=[dfG['party'][book] for book in books], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    margin=dict(b=110),
    #barmode='stack',
    xaxis=dict(title_text='book'),
    yaxis=dict(title_text='encounters', dtick=50, minor_dtick=25),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [392]:
# histogram of total encounters by book
import plotly.graph_objects as go

dfG = df[['book_acronym','party_level','party']].groupby(['book_acronym','party_level'], observed=True).count().reset_index()
dfG = dfG.groupby('book_acronym', observed=True).median()

books = [v['acronym'] for k, v in book_dict.items()]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=books,
    y=[dfG['party'][book] for book in books], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #margin=dict(b=110),
    #barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters per level', automargin=True, dtick=5, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [215]:
# histogram of total encounters by book
import plotly.graph_objects as go

dft = df[df['2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['party_level','party']].groupby(['party_level'], observed=True).count().reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=dfG['party_level'],
    y=dfG['party'], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #margin=dict(b=110),
    #barmode='stack',
    xaxis=dict(title_text='level'),
    yaxis=dict(title_text='encounters', range=[0,400], dtick=50, minor_dtick=25),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

## Encounter Difficulties (2014)

In [6]:
# histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dft = df[~df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

"""fig.add_trace(go.Bar(
    x=dfG[f'{rules} difficulty'],
    y=dfG[f'{rules} XP_ratio']/dfG[f'{rules} XP_ratio'].sum(), 
    hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    #xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.40], tickformat='.0%', dtick=0.05, minor_dtick=0.01),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-2014-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-2014-small')

In [7]:
# Plots a histogram of encounter XPs using 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()
ymax = 0.16
dft = df[df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
fig.add_trace(go.Histogram(x=dft[f'{rules} XP_ratio'], xbins_size=0.05, histnorm='probability', showlegend=False))

if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[diff, diff+dx], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=diff + dx/2, 
        y=0.9*ymax,
        text=name,
        showarrow=False,
        font_size=12,
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='XP ratio', range=[0,1.5], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='encounters [%]', range=[0,ymax], tickformat='.0%', dtick=0.02, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-pdf-2014-small', style='width: 600px;')
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-pdf-2014-large', style='width: 600px;')

In [8]:
# Plots a histogram of encounter XPs using 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

#dft = df[df['2014 difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly'])]
dft = df[~df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dft = dft[dft['2014 XP_ratio'].between(0, 2)]
vals, edges = np.histogram(dft['2014 XP_ratio'], 40)

x = np.convolve(edges, np.ones(2)/2, mode='valid')
y = vals/sum(vals)
fig.add_trace(go.Scatter(x=x, y=y*x, mode='markers+lines', name='pre-2022'))

dft = df[df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dft = dft[dft['2014 XP_ratio'].between(0, 2)]
vals, edges = np.histogram(dft['2014 XP_ratio'], 40)

x = np.convolve(edges, np.ones(2)/2, mode='valid')
y = vals/sum(vals)
fig.add_trace(go.Scatter(x=x, y=y*x, mode='markers+lines', name='post-2022'))



# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='XP ratio', range=[0,1.5], dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='encounters', range=[0,0.04]),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-pdf-2014-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-pdf-2014-small', style='width: 600px;')

In [9]:
# histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dft = df[df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
books = df['book_acronym'].unique()

dfG = dft[['book_acronym',f'{rules} XP_ratio']].groupby('book_acronym', observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}
print(encounter_counts)

fig = go.Figure()
diff_list = list(dft[f'{rules} difficulty'].cat.categories)
for difficulty in dft[f'{rules} difficulty'].unique():
    print(difficulty)
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = dft[dft[f'{rules} difficulty'].eq(difficulty)]
    dfG = df1[[f'{rules} difficulty','book_acronym']].groupby(['book_acronym'], observed=True).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['book_acronym']], axis=1)
    dfG = dfG[dfG['book_acronym'].isin(books)]
    
    fig.add_trace(go.Bar(
        x=dfG['book_acronym'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='%{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=550,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-by-book-2014-large', style='aspect-ratio: 600/550;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-by-book-2014-small', style='aspect-ratio: 600/550;')

{'ToD': np.int64(140), 'PotA': np.int64(127), 'CoS': np.int64(78), 'SKT': np.int64(168), 'TftYP': np.int64(292), 'ToA': np.int64(119), 'W:DH': np.int64(95), 'W:DotMM': np.int64(276), 'GoS': np.int64(110), 'BG:DiA': np.int64(105), 'ID:RotF': np.int64(133), 'CM': np.int64(87), 'CR:CotN': np.int64(67), 'JttRC': np.int64(62), 'D:SotDQ': np.int64(106), 'KftGV': np.int64(83), 'PaB:TSO': np.int64(126), 'V:EoR': np.int64(99), 'QftIS': np.int64(189), 'DD': np.int64(81)}
Easy
Medium
Hard
Deadly
Very Deadly


In [10]:
# histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dfG = df[['book_acronym',f'{rules} XP_ratio']].groupby('book_acronym', observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}

#books = ['ToD','PotA','SKT','ToA','W:DH','W:DotMM','BG:DiA','ID:RotF','CR:CotN','D:SotDQ','PaB:TSO','V:EoR']
books = df['book_acronym'].unique()

fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)
for difficulty in df[f'{rules} difficulty'].cat.categories:
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = df[df[f'{rules} difficulty'].eq(difficulty)]
    dfG = df1[[f'{rules} difficulty','book_acronym']].groupby(['book_acronym'], observed=False).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['book_acronym']], axis=1)
    dfG = dfG[dfG['book_acronym'].isin(books)]
    
    fig.add_trace(go.Bar(
        x=dfG['book_acronym'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='%{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=550,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-by-book-2014-large', style='aspect-ratio: 600/550;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-by-book-2014-small', style='aspect-ratio: 600/550;')

In [11]:
# histogram of difficulties by level using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dft = df[df[f'{rules} difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['party_level',f'{rules} XP_ratio']].groupby(['party_level'], observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}

# color turbo
fig = go.Figure()
diff_list = list(dft[f'{rules} difficulty'].cat.categories)
for difficulty in dft[f'{rules} difficulty'].cat.categories:
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = dft[dft[f'{rules} difficulty'].eq(difficulty)]
    if len(df1) == 0: continue
    dfG = df1[[f'{rules} difficulty','party_level']].groupby(['party_level'], observed=True).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['party_level']], axis=1)
    
    fig.add_trace(go.Bar(
        x=dfG['party_level'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='level %{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='party level', automargin=True, dtick=5, minor_dtick=1),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-by-level-2014-large', style='width: 600px;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-by-level-2014-small', style='width: 600px;')

In [12]:
# histogram of difficulties by level using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2014'

dft = df[df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['party_level',f'{rules} XP_ratio']].groupby(['party_level'], observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}

# color turbo
fig = go.Figure()
diff_list = list(dft[f'{rules} difficulty'].cat.categories)
for difficulty in dft[f'{rules} difficulty'].cat.categories:
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = dft[dft[f'{rules} difficulty'].eq(difficulty)]
    if len(df1) == 0: continue
    dfG = df1[[f'{rules} difficulty','party_level']].groupby(['party_level'], observed=True).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['party_level']], axis=1)
    
    fig.add_trace(go.Bar(
        x=dfG['party_level'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='level %{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='party level', automargin=True, dtick=5, minor_dtick=1),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-by-level-2014-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-by-level-2014-small', style='width: 600px;')

## Encounter Difficulties (2024)

In [13]:
# Plots a histogram of encounter XPs using 2024 rules
import plotly.graph_objects as go

rules = '2024'

fig = go.Figure()
ymax = 0.16
fig.add_trace(go.Histogram(x=df[f'{rules} XP_ratio'], xbins_size=0.05, histnorm='probability', showlegend=False))

if rules == '2014':
    thresholds = [
        ('Trivial',     0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',        0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',      0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',        0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',      0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly', 1.00, 1.00, 'rgba(200, 200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 0.80, 'rgba(200, 200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[diff, diff+dx], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=diff + dx/2, 
        y=0.9*ymax,
        text=name,
        showarrow=False,
        font_size=12,
    )

"""x = np.linspace(0.15, 2.0, 100)
fig.add_trace(go.Scatter(
    x=x, 
    y=50*(0.15/x),
    mode='lines',
    showlegend=False,
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='encounter XP ratio', range=[0,1.5], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='percent', range=[0,ymax], tickformat='.0%', dtick=0.02, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-pdf-2024-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-pdf-2024-small', style='width: 600px;')

In [14]:
# histogram of difficulties by book using the 2024 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2024'

dft = df[df['2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

"""fig.add_trace(go.Bar(
    x=dfG[f'{rules} difficulty'],
    y=dfG[f'{rules} XP_ratio']/dfG[f'{rules} XP_ratio'].sum(), 
    hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    #xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.90], tickformat='.0%', dtick=0.10, minor_dtick=0.02),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-2024-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-2024-small')

In [15]:
# histogram of difficulties by book using the 2014 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2024'

dft = df[df['2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
dfG = dft[['book_acronym',f'{rules} XP_ratio']].groupby('book_acronym', observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}

#books = ['ToD','PotA','SKT','ToA','W:DH','W:DotMM','BG:DiA','ID:RotF','CR:CotN','D:SotDQ','PaB:TSO','V:EoR']
books = df['book_acronym'].unique()

fig = go.Figure()
diff_list = list(dft[f'{rules} difficulty'].cat.categories)
for difficulty in dft[f'{rules} difficulty'].cat.categories:
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = dft[dft[f'{rules} difficulty'].eq(difficulty)]
    dfG = df1[[f'{rules} difficulty','book_acronym']].groupby(['book_acronym'], observed=False).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['book_acronym']], axis=1)
    dfG = dfG[dfG['book_acronym'].isin(books)]
    
    fig.add_trace(go.Bar(
        x=dfG['book_acronym'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='%{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=550,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-by-book-2024-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-by-book-2024-small', style='width: 600px;')

In [16]:
# histogram of difficulties by level using the 2024 rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = '2024'

dfG = df[['party_level',f'{rules} XP_ratio']].groupby(['party_level'], observed=True).count()
encounter_counts = {b: dfG[f'{rules} XP_ratio'][b] for b in dfG.index}

# color turbo
fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)
for difficulty in df[f'{rules} difficulty'].cat.categories:
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]
    df1 = df[df[f'{rules} difficulty'].eq(difficulty)]
    if len(df1) == 0: continue
    dfG = df1[[f'{rules} difficulty','party_level']].groupby(['party_level'], observed=False).count().reset_index()
    dfG[f'{rules} difficulty'] = dfG.apply(lambda r: r[f'{rules} difficulty']/encounter_counts[r['party_level']], axis=1)
    
    fig.add_trace(go.Bar(
        x=dfG['party_level'],
        y=dfG[f'{rules} difficulty'], 
        name=difficulty,
        marker_color=color,
        hovertemplate='level %{x}<br>' + f'{difficulty}<br>'+'%{y:.1%}<extra></extra>',
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='party level', automargin=True, dtick=5, minor_dtick=1),
    yaxis=dict(title_text='encounters (%)', range=[0,1.0], tickformat='.0%', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [17]:
# histogram of difficulties by book using the 2024 rules split about 2022
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale
from plotly.subplots import make_subplots

rules = '2024'

diff_list = list(df[f'{rules} difficulty'].cat.categories)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=['pre-2022','post-2022'])

# pre-2022
dft = df[~df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ), row=1, col=1)

# post-2022
dft = df[df['book_acronym'].isin(['CR:CotN','JttRC','D:SotDQ','KftGV','PaB:TSO','V:EoR','QftIS','DD'])]
dfG = dft[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ), row=1, col=2)


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict( automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.9], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis2=dict(dtick=0.1, minor_dtick=0.05),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=800, 
    height=450,
)

for a in fig.layout.annotations:
    a.update(y=a['y']-0.08)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-split-2014-large', style='aspect-ratio: 800/450;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-split-2014-small', style='aspect-ratio: 800/450;')

## Encounter Difficulties (TFB)

In [18]:
# histogram of difficulties by book using the TFB rules
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

rules = 'TFB'

dfG = df[[f'{rules} difficulty',f'{rules} XP_ratio']].groupby(f'{rules} difficulty', observed=True).count().reset_index()

fig = go.Figure()
diff_list = list(df[f'{rules} difficulty'].cat.categories)

n_difficulties = len(dfG[f'{rules} difficulty'])
for i in range(n_difficulties):
    difficulty = dfG[f'{rules} difficulty'][i]
    color = sample_colorscale('YlOrRd', (diff_list.index(difficulty) + 0.5)/len(diff_list))[0]

    fig.add_trace(go.Bar(
        x=dfG[f'{rules} difficulty'],
        y=[dfG[f'{rules} XP_ratio'][j] if i ==j else 0 for j in range(n_difficulties)]/dfG[f'{rules} XP_ratio'].sum(), 
        name=difficulty,
        marker_color=color,
        showlegend=False,
        text=['{0:.1%}'.format(dfG[f'{rules} XP_ratio'][j]/dfG[f'{rules} XP_ratio'].sum()) if i ==j else '' for j in range(n_difficulties)],
        #texttemplate="%{y} x %{width} =<br>%{customdata[1]}",
        textposition="outside",
        textangle=0,
        textfont_color="black",
        hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
    ))

"""fig.add_trace(go.Bar(
    x=dfG[f'{rules} difficulty'],
    y=dfG[f'{rules} XP_ratio']/dfG[f'{rules} XP_ratio'].sum(), 
    hovertemplate='%{x}<br>%{y:.1%}<extra></extra>',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    #xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='encounters [%]', range=[0,0.40], tickformat='.0%', dtick=0.05, minor_dtick=0.01),
    #legend=dict(xanchor='left', x=0.0, yanchor='bottom', y=1.04, orientation='h'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-2014-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-2014-small')

In [19]:
# Plots a histogram of encounter XPs using TFB rules
import plotly.graph_objects as go

rules = 'TFB'

fig = go.Figure()
ymax = 0.16
fig.add_trace(go.Histogram(x=df[f'{rules} XP_ratio'], xbins_size=0.05, histnorm='probability', showlegend=False))

if rules in ['2014','TFB']:
    thresholds = [
        ('Trivial',     0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',        0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',      0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',        0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',      0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly', 1.00, 0.50, 'rgba(200, 200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 0.80, 'rgba(200, 200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[diff, diff+dx], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=diff + dx/2, 
        y=0.9*ymax,
        text=name,
        showarrow=False,
        font_size=12,
    )

"""x = np.linspace(0.15, 2.0, 100)
fig.add_trace(go.Scatter(
    x=x, 
    y=50*(0.15/x),
    mode='lines',
    showlegend=False,
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='encounter XP ratio', range=[0,1.5], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='percent', range=[0,ymax], tickformat='.0%', dtick=0.02, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-difficulty-pdf-2024-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-difficulty-pdf-2024-small', style='width: 600px;')

## Encounter XP Ratio (2014)

In [20]:
# Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 1.0, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[-1, len(books)+2], 
        y=[diff+dx, diff+dx],
        mode='none',
        fill='tonexty', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=len(books)+1, 
        y=diff + dx/2,
        text=name,
        showarrow=False,
        font_size=12,
    )


for i in range(len(books)):
    book = books[i]
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    #print(f'{book}: {xr_mean:.3f} ({xr_std:.3f}) --> {xr_std/xr_mean:.3f}')
    dfG = df1[['book_acronym','party_level','adventure',f'{rules} XP_ratio']].groupby(['book_acronym','party_level','adventure'], observed=True).median().reset_index()
    fig.add_trace(go.Scatter(
        x=tfb.jitter([i]*len(dfG[f'{rules} XP_ratio']), 0.15), 
        y=dfG[f'{rules} XP_ratio'],
        mode='markers',
        name=dfG['book_acronym'][0],
        showlegend=False,
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [21]:
# Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

encounter_cvs = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounter_cvs.append(xr_cv)

fig.add_trace(go.Scatter(
    x=books, 
    y=encounter_cvs,
    mode='markers',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio CV', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [22]:
# Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2024'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

encounter_cvs = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounter_cvs.append(xr_cv)

fig.add_trace(go.Scatter(
    x=books, 
    y=encounter_cvs,
    mode='markers',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio CV', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [23]:
# Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

encounters_mom = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Trivial','Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_med = df1[f'{rules} XP_ratio'].median()
    encounters_mom.append(xr_med/xr_mean)

fig.add_trace(go.Scatter(
    x=books, 
    y=encounters_mom,
    mode='markers',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio median/mean', range=[0,1.5], automargin=True, tickformat='.1f', dtick=0.2, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [24]:
# Plots the distribution of encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go
import numpy as np

rules = '2014'

SHOW_THRESHOLDS = True
fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

y_range = None
if SHOW_THRESHOLDS:
    y_range = [0,3]
    if rules == '2014':
        thresholds = [
            ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
            ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
            ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
            ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
            ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
            ('Very Deadly',  1.0, max(1.0,max(y_range)-1), 'rgba(200,  200, 200, 0.15)'),
        ]
    else:
        thresholds = [
            ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
            ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
            ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
            ('Very High', 0.70, max(1.0,max(y_range)-0.7), 'rgba(200,  200, 200, 0.15)'),
        ]
    for name, diff, dx, color in thresholds:
        fig.add_trace(go.Scatter(
            x=[-1, len(books)+3], 
            y=[diff+dx, diff+dx],
            mode='none',
            fill='tonexty', 
            showlegend=False,
            fillcolor=color,
            hoverinfo='none'
        ))
        fig.add_annotation(
            x=len(books)+1, 
            y=diff + dx/2,
            text=name,
            showarrow=False,
            font_size=12,
        )

colors = iter(COLOR_LIST)
for i in range(len(books)):
    book = books[i]
    df1 = df[df['book_acronym'].eq(book)]
    fig.add_trace(go.Scatter(
        x=tfb.jitter([i]*len(df1[f'{rules} XP_ratio']), 0.10), 
        y=df1[f'{rules} XP_ratio'],
        mode='markers',
        name=book,
        marker_size=3,
        line_color=next(colors),
        showlegend=False,
    ))

# set layout
if SHOW_THRESHOLDS:
    x_range = [-1,len(books)+1]
else:
    x_range = [-1,len(books)]

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book', automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio', range=y_range, automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

## Encounter XP Ratio (2024)

In [25]:
# Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2024'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 1.0, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[-1, len(books)+2], 
        y=[diff+dx, diff+dx],
        mode='none',
        fill='tonexty', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=len(books)+1, 
        y=diff + dx/2,
        text=name,
        showarrow=False,
        font_size=12,
    )


for i in range(len(books)):
    book = books[i]
    df1 = df[df['book_acronym'].eq(book) & df[f'{rules} difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly','Low','Moderate','High','Very High'])]
    xr_mean = df1[f'{rules} XP_ratio'].mean()
    xr_std = df1[f'{rules} XP_ratio'].std()
    #print(f'{book}: {xr_mean:.3f} ({xr_std:.3f}) --> {xr_std/xr_mean:.3f}')
    dfG = df1[['book_acronym','party_level','adventure',f'{rules} XP_ratio']].groupby(['book_acronym','party_level','adventure'], observed=True).mean().reset_index()
    fig.add_trace(go.Scatter(
        x=tfb.jitter([i]*len(dfG[f'{rules} XP_ratio']), 0.15), 
        y=dfG[f'{rules} XP_ratio'],
        mode='markers',
        name=dfG['book_acronym'][0],
        showlegend=False,
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [26]:
# Plots the distribution of encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go
import numpy as np

rules = '2024'

SHOW_THRESHOLDS = True
fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]

y_range = None
if SHOW_THRESHOLDS:
    y_range = [0,3]
    if rules == '2014':
        thresholds = [
            ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
            ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
            ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
            ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
            ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
            ('Very Deadly',  1.0, max(1.0,max(y_range)-1), 'rgba(200,  200, 200, 0.15)'),
        ]
    else:
        thresholds = [
            ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
            ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
            ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
            ('Very High', 0.70, max(1.0,max(y_range)-0.7), 'rgba(200,  200, 200, 0.15)'),
        ]
    for name, diff, dx, color in thresholds:
        fig.add_trace(go.Scatter(
            x=[-1, len(books)+3], 
            y=[diff+dx, diff+dx],
            mode='none',
            fill='tonexty', 
            showlegend=False,
            fillcolor=color,
            hoverinfo='none'
        ))
        fig.add_annotation(
            x=len(books)+1, 
            y=diff + dx/2,
            text=name,
            showarrow=False,
            font_size=12,
        )

colors = iter(COLOR_LIST)
for i in range(len(books)):
    book = books[i]
    df1 = df[df['book_acronym'].eq(book)]
    fig.add_trace(go.Scatter(
        x=tfb.jitter([i]*len(df1[f'{rules} XP_ratio']), 0.10), 
        y=df1[f'{rules} XP_ratio'],
        mode='markers',
        name=book,
        marker_size=3,
        line_color=next(colors),
        showlegend=False,
    ))

# set layout
if SHOW_THRESHOLDS:
    x_range = [-1,len(books)+1]
else:
    x_range = [-1,len(books)]

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='book', automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio', range=y_range, automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

## Encounter XP ratio comparison

In [27]:
# Fig. 4: Plots the distribution of average encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go

rules = '2014'

fig = go.Figure()

books = [v['acronym'] for k, v in book_dict.items()]


if rules == '2014':
    thresholds = [
        ('Trivial', 0.00, 0.15, 'rgba(167, 167, 167, 0.15)'),
        ('Easy',    0.15, 0.15, 'rgba(  0, 199, 151, 0.15)'),
        ('Medium',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('Hard',    0.45, 0.25, 'rgba(228,  90,  29, 0.15)'),
        ('Deadly',  0.70, 0.30, 'rgba(144,  79, 213, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very Deadly',  1.0, 0.5, 'rgba(200,  200, 200, 0.15)'),
    ]
else:
    thresholds = [
        ('Low',       0.00, 0.30, 'rgba(  0, 199, 151, 0.15)'),
        ('Moderate',  0.30, 0.15, 'rgba(245, 166,  35, 0.15)'),
        ('High',      0.45, 0.25, 'rgba(228,  90,  29, 0.15)'), #rgba(213,  79,  79, 0.2)'
        ('Very High', 0.70, 1.30, 'rgba(200,  200, 200, 0.15)'),
    ]
for name, diff, dx, color in thresholds:
    fig.add_trace(go.Scatter(
        x=[-1, len(books)+2], 
        y=[diff+dx, diff+dx],
        mode='none',
        fill='tonexty', 
        showlegend=False,
        fillcolor=color,
        hoverinfo='none'
    ))
    fig.add_annotation(
        x=len(books)+1, 
        y=diff + dx/2,
        text=name,
        showarrow=False,
        font_size=12,
    )

encounters_mean = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
    xr_mean = df1[f'2014 XP_ratio'].median()
    xr_std = df1[f'2014 XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounters_mean.append(xr_mean)

fig.add_trace(go.Scatter(
    x=list(range(len(encounters_mean))), 
    y=encounters_mean,
    mode='markers+lines',
    name='2014 rules',
    line_color=COLOR_LIST[0],
))

encounters_mean = []
for book in books:
    df1 = df[df['book_acronym'].eq(book) & df[f'2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]
    xr_mean = df1[f'2024 XP_ratio'].median()
    xr_std = df1[f'2024 XP_ratio'].std()
    xr_cv = xr_std/xr_mean
    encounters_mean.append(xr_mean)

fig.add_trace(go.Scatter(
    x=list(range(len(encounters_mean))), 
    y=encounters_mean,
    mode='markers+lines',
    name='2024 rules',
    line_color=COLOR_LIST[1]
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='XP ratio mean', range=[0,1.0], automargin=True, tickformat='.1f', dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.02, yanchor='top', y=1.00, bgcolor='rgba(0,0,0,0)'),
    width=600, 
    height=500,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-encounter-median-xp-ratio-by-book-large', style='aspect-ratio: 600/500;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-encounter-median-xp-ratio-by-book-small', style='aspect-ratio: 600/500;')

In [28]:
# Plots XP ratio using the 2024 rules against the 2014 rules (averaged by book and level)
import plotly.graph_objects as go

fig = go.Figure()

dfG = df[['party_level','book','2014 XP_ratio','2024 XP_ratio']].groupby(['party_level','book'], observed=True).mean()
fig.add_trace(go.Scatter(
    x=dfG['2014 XP_ratio'], 
    y=dfG['2024 XP_ratio'],
    mode='markers',
    marker_size=4,
    showlegend=False,
))

fig.add_trace(go.Scatter(
    x=[0,2], 
    y=[0,2],
    mode='lines',
    line_color='black',
    line_dash='dash',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='2014 XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='2024 XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [29]:
# Plots XP ratio using the 2024 rules against the 2014 rules (averaged by book and level)
import plotly.graph_objects as go

fig = go.Figure()

dfG = df[['party_level','book','2014 XP_ratio','2024 XP_ratio','TFB XP_ratio']].groupby(['party_level','book'], observed=True).mean()
fig.add_trace(go.Scatter(
    x=dfG['2014 XP_ratio'], 
    y=dfG['2024 XP_ratio'],
    mode='markers',
    marker_size=4,
    name='2024 vs 2014',
))

fig.add_trace(go.Scatter(
    x=dfG['2014 XP_ratio'], 
    y=dfG['TFB XP_ratio'],
    mode='markers',
    marker_size=4,
    name='TFB vs 2014',
))

fig.add_trace(go.Scatter(
    x=[0,2], 
    y=[0,2],
    mode='lines',
    line_color='black',
    line_dash='dash',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='2014 XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='2024 XP ratio', range=[0,2], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [30]:
# Plots XP ratio using the 2024 rules against the 2014 rules (averaged by book and level)
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['2014 XP_ratio'], 
    y=df['2024 XP_ratio'],
    customdata=df['id'],
    mode='markers',
    marker_size=4,
    name='2024 vs 2014',
    hovertemplate='id %{customdata}<br>2014 XP ratio %{x:.3f}<br>2024 XP ratio %{y:.3f}<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=df['2014 XP_ratio'], 
    y=df['TFB XP_ratio'],
    customdata=df['id'],
    mode='markers',
    marker_size=4,
    name='TFB vs 2014',
    hovertemplate='id %{customdata}<br>2014 XP ratio %{x:.3f}<br>TFB XP ratio %{y:.3f}<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=[0,8], 
    y=[0,8],
    mode='lines',
    line_color='black',
    line_dash='dash',
    showlegend=False,
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='2014 XP ratio', range=[0,8], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='2024 XP ratio', range=[0,8], automargin=True, tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

"""
encounters that are harder with TFB rules are ones with more than 16 monsters in them.
"""
fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

## Adventuring Days (2014)

In [31]:
# Plots adventuring days vs level
import plotly.graph_objects as go
import numpy as np

rules = '2014'

books = [v['acronym'] for k, v in book_dict.items()]

fig = go.Figure()

dfG = df[['book_acronym','party_level','adventure',f'{rules} XP_ratio']].groupby(['book_acronym','party_level','adventure'], observed=True).sum().reset_index()
dfG = dfG[['party_level',f'{rules} XP_ratio']].groupby(['party_level']).median().reset_index()

fig.add_trace(go.Scatter(
    x=dfG['party_level'], 
    y=dfG[f'{rules} XP_ratio']/2, 
    mode='markers',
    hovertemplate = 'level %{x:.0f}<br>days %{y:.2f}<extra></extra>',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='adventuring days', range=[0,4], tickformat='.1f'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-adventuring-days-vs-level-2014-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-adventuring-days-vs-level-2014-small')

In [32]:
# Plots adventuring days by level for each book
import plotly.graph_objects as go
import numpy as np

rules = '2014'

books = [v['acronym'] for k, v in book_dict.items()]

fig = go.Figure()

dfG = df[['book_acronym','party_level','adventure',f'{rules} XP_ratio']].groupby(['book_acronym','party_level','adventure'], observed=True).sum().reset_index()
dfG = dfG[['book_acronym',f'{rules} XP_ratio']].groupby(['book_acronym']).median()

fig.add_trace(go.Scatter(
    x=books, 
    y=[dfG[f'{rules} XP_ratio'][book]/2 for book in books], 
    mode='markers+lines',
    hovertemplate = '%{x}<br>days %{y:.2f}<extra></extra>',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='book',range=[-1,len(books)+2], automargin=True, tickmode='array', tickvals=list(range(len(books))), ticktext=list(books)),
    yaxis=dict(title_text='adventuring days', range=[0,4], tickformat='.1f'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-adventuring-days-vs-level-2014-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-adventuring-days-vs-level-2014-small')

## Adventuring Days (2024)

In [33]:
# Plots adventuring days vs level
import plotly.graph_objects as go
import numpy as np

rules = '2024'

fig = go.Figure()
bins = np.linspace(0,2,21)
ymax = 0
dfG = df[['book_acronym','party_level','adventure',f'{rules} XP_ratio']].groupby(['book_acronym','party_level','adventure'], observed=True).sum().reset_index()
dfG = dfG[['party_level',f'{rules} XP_ratio']].groupby(['party_level']).median().reset_index()

fig.add_trace(go.Scatter(
    x=dfG['party_level'], 
    y=dfG[f'{rules} XP_ratio']/2, 
    mode='markers',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='adventuring days', range=[0,3], tickformat='.1f'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-adventuring-days-vs-level-2024-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-adventuring-days-vs-level-2024-small')

## Monsters per Encounter

In [34]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()
ymax = 0.35
"""xbins=dict(
        start=-3.0,
        end=4,
        size=0.5
    )"""
fig.add_trace(go.Histogram(
    x=df['n_monsters'], 
    xbins_size=1, 
    histnorm='probability', 
    showlegend=False,
    hovertemplate = ''
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='monsters per encounter', range=[0,20], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='encounter (%)', range=[0,ymax], tickformat='.0%', dtick=0.05, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [35]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()
ymax = 0.75
"""xbins=dict(
        start=-3.0,
        end=4,
        size=0.5
    )"""
fig.add_trace(go.Histogram(x=df['n_unique_monsters'], xbins_size=1, histnorm='probability', showlegend=False))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='unique monsters per encounter', range=[0,7], tickformat='.0f', dtick=1, minor_dtick=1),
    yaxis=dict(title_text='encounter (%)', range=[0,ymax], tickformat='.0%', dtick=0.10, minor_dtick=0.02),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [36]:
# histogram of difficulties by book
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

#books = ['ToD','PotA','SKT','ToA','W:DotMM','BG:DiA','ID:RotF','CR:CotN','D:SotDQ','PaB:TSO','V:EoR']
books = df['book_acronym'].unique()

fig = go.Figure()

unique_monsters = []
for book in books:
    dft = df[df['book_acronym'].eq(book)]
    monsters = []
    for x in dft['monsters']:
        monsters.extend(x)
    unique_monsters.append(len(np.unique(monsters)))

fig.add_trace(go.Bar(
    x=books,
    y=unique_monsters, 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='unique monsters', automargin=True, tickformat='.0f', dtick=50, minor_dtick=10),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [37]:
# Plots the average number of monsters per encounter by level
import plotly.graph_objects as go

levels = list(range(1,21))

fig = go.Figure()

dft = df[df['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
dft = dft[['party_level','n_monsters']].groupby(['party_level']).mean().reset_index()


fig.add_trace(go.Scatter(
    x=dft['party_level'],
    y=dft['n_monsters'],
    showlegend=False,
    mode='markers',
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [38]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

dft = df[df['2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).quantile(0.2).reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    line=dict(color=COLOR_LIST[0], width=0),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).quantile(0.8).reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    fill='tonexty',
    line=dict(color=COLOR_LIST[0], width=0),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).median().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    line=dict(color=COLOR_LIST[0], width=2),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,10], tickformat='.0f', dtick=2, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [39]:
# Plots a histogram of encounter XPs by book
import plotly.graph_objects as go

ymax = 0.35

fig = go.Figure()

for diff in ['Easy','Medium','Hard','Deadly','Very Deadly']:
    df1 = df[df['2014 difficulty'].eq(diff)]
    y, x = np.histogram(df1['n_monsters'], bins=np.linspace(1,21,21)-0.5)
    x = (x[1:] + x[0:-1])/2
    fig.add_trace(go.Scatter(
        x=x, 
        y=y/sum(y),
        #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
        mode='markers+lines',
        name=diff,
        showlegend=True,
        hovertemplate = f'<b>{diff}</b><br>'
                + 'monsters %{x:.0f}<br>'
                + 'probability %{y:.1%}'
                + '<extra></extra>'
    ))

y, x = np.histogram(df['n_monsters'], bins=np.linspace(1,21,21)-0.5)
x = (x[1:] + x[0:-1])/2
fig.add_trace(go.Scatter(
    x=x, 
    y=y/sum(y),
    #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
    mode='markers+lines',
    name='average',
    line_color='black',
    line_dash='dash',
    showlegend=True,
    hovertemplate = f'<b>Average</b><br>'
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monsters per encounter', range=[0,10], dtick=1, minor_dtick=1),
    yaxis=dict(title_text='encounters [%]', range=[-0.01, 0.40], tickformat='.0%'),
    legend=dict(
        xanchor='right', x=1.00, 
        yanchor='top', y=1.00,
        orientation='v',
    ),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-by-difficulty-large', style='width: 600px;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-by-difficulty-small', style='width: 600px;')

## Median monster XP by level

In [45]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,18))

#fig = go.Figure()
fig = make_subplots(rows=len(levels), cols=1, shared_xaxes=True, vertical_spacing=0.005, subplot_titles=[f'level {x}' for x in levels])


def x_decoder(x):
    match x:
        case 0:
            return 0
        case 0.125:
            return 0.125
        case 0.25:
            return 0.28125
        case 0.5:
            return 0.5625
        case 1:
            return 1.125
        case _:
            return x

def w_decoder(x):
    match x:
        case 0:
            return 0.125
        case 0.125:
            return 0.125
        case 0.25:
            return 0.1875
        case 0.5:
            return 0.375
        case 1:
            return 0.75
        case _:
            return 1.0


for level in levels:
    #dft = df[df['party_level'].eq(level) & df['2014 XP_ratio'].ge(0.30)]
    dft = df[df['party_level'].eq(level) & df['2014 difficulty'].isin(['Medium','Hard','Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    #fig.add_trace(go.Histogram(x=monster_cr_values, xbins_size=1, histnorm='probability', showlegend=False))

    unique_cr_values = np.unique(monster_cr_values)
    c = [np.sum(np.equal(monster_cr_values, cr))/len(monster_cr_values) for cr in unique_cr_values]
    x = [x_decoder(cr) for cr in unique_cr_values]
    w = [w_decoder(cr) for cr in unique_cr_values]
    fig.add_trace(go.Bar(
        x=x,
        y=c,
        width=w,
        showlegend=False,
        #customdata=dfC.index,
        #hovertemplate = '<b>' + g + '</b><br>'
        #    + 'CR %{customdata}<br>'
        #    + 'Monsters %{y:,.0f}'
        #    + '<extra></extra>'
    ), row=int(level), col=1)
    fig.add_trace(go.Scatter(
        x=[level,level],
        y=[0,max(c)],
        mode='lines',
        line_color='black',
        line_dash='dash',
        showlegend=False,
        hoverinfo='skip',
    ), row=int(level), col=1)


layout = {}
for level in levels: 
    if level == 1:
        layout['yaxis'] = dict(title_text='monsters (%)', tickformat='.0%', dtick=0.10, minor_dtick=0.05)
    else:
        layout[f'yaxis{level:.0f}'] = dict(title_text='monsters (%)', tickformat='.0%', dtick=0.10, minor_dtick=0.05)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis20=dict(title_text='challenge rating', range=[0,30], tickformat='.0f'),
    **layout,
    width=550, 
    height=150*20,
)
for a in fig.layout.annotations:
    a.update(y=a['y']-0.006)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [41]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

yMed = []
yHigh = []
yLow  = []
for level in levels:
    #dft = df[df['party_level'].eq(level) & df['2014 XP_ratio'].ge(0.30)]
    dft = df[df['party_level'].eq(level) & df['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    yMed.append(np.median(monster_cr_values))
    yHigh.append(np.quantile(monster_cr_values, 0.8))
    yLow.append(np.quantile(monster_cr_values, 0.2))


fig.add_trace(go.Scatter(
    x=levels,
    y=yLow,
    showlegend=False,
    mode='lines',
    line_width=0,
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yHigh,
    showlegend=False,
    mode='lines',
    line_width=0,
    fill='tonexty',
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yMed,
    showlegend=False,
    mode='lines',
    line_color=COLOR_LIST[0],
    #customdata=dfC.index,
    hovertemplate = ''
        + 'level %{x:.0f}<br>'
        + 'CR %{y:.0f}'
        + '<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[0,21],
    y=[0,21],
    showlegend=False,
    mode='lines',
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='challenge rating', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-cr-range-vs-level-large', selector={'name': 'none'})
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-cr-range-vs-level-small', selector={'name': 'none'})

In [42]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

yMed = []
yHigh = []
yLow  = []
for level in levels:
    #dft = df[df['party_level'].eq(level) & df['2014 XP_ratio'].ge(0.30)]
    dft = df[df['party_level'].eq(level) & df['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
    monster_xp_values = []
    for x in dft['monsters_xp']:
        monster_xp_values.extend(x)

    yMed.append(np.median(monster_xp_values)/(0.5*player_character_xp_budget(level)))
    yHigh.append(np.quantile(monster_xp_values, 0.8)/(0.5*player_character_xp_budget(level)))
    yLow.append(np.quantile(monster_xp_values, 0.2)/(0.5*player_character_xp_budget(level)))

fig.add_trace(go.Scatter(
    x=levels,
    y=yLow,
    showlegend=False,
    mode='lines',
    line_width=0,
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yHigh,
    showlegend=False,
    mode='lines',
    line_width=0,
    fill='tonexty',
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yMed,
    showlegend=False,
    mode='lines',
    line_color=COLOR_LIST[0],
    #customdata=dfC.index,
    hovertemplate = ''
        + 'level %{x:.0f}<br>'
        + 'XP ratio %{y:.3f}'
        + '<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[1,20],
    y=[1.2,1.2],
    showlegend=False,
    mode='lines',
    line_dash='dash',
    line_color='black',
    hoverinfo='skip',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster XP ratio', range=[0,1.5], tickformat='.1f'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-xp-range-vs-level-large', selector={'name': 'none'})
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-xp-range-vs-level-small', selector={'name': 'none'})

In [43]:
# Plots the distribution of encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go
import numpy as np

rules = '2014'

fig = go.Figure()

colors = iter(COLOR_LIST)
for lvl in df['party_level'].unique():
    dft = df[df['party_level'].eq(lvl) & df['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    fig.add_trace(go.Scatter(
        x=tfb.jitter([lvl]*len(monster_cr_values), 0.10), 
        y=tfb.jitter(monster_cr_values, 0.10), 
        mode='markers',
        name=book,
        marker_size=3,
        line_color=COLOR_LIST[0],
        showlegend=False,
    ))

fig.add_trace(go.Scatter(
    x=[0,21],
    y=[0,21],
    showlegend=False,
    mode='lines',
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='challenge rating'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [44]:


# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

yMed = []
yHigh = []
yLow  = []
for level in levels:
    #dft = df[df['party_level'].eq(level) & df['2014 XP_ratio'].ge(0.30)]
    dft = df[df['party_level'].eq(level) & df['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    [dmg5e.monster_default_stats(cr)['AC'] for cr in monster_cr_values]